# фильтрация артефактов ЭМГ

## импорты

In [ ]:
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy import linalg, signal

mne.set_log_level("WARNING")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.grid"] = True


## пути

In [ ]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_raw_dir = project_root / "data" / "raw"
data_interim_dir = project_root / "data" / "interim"
data_processed_dir = project_root / "data" / "processed"
figures_dir = project_root / "outputs" / "figures"
qc_dir = project_root / "outputs" / "qc"
tables_dir = project_root / "outputs" / "tables"

for folder in [data_interim_dir, data_processed_dir, figures_dir, qc_dir, tables_dir]:
    folder.mkdir(parents=True, exist_ok=True)

fif_path = data_raw_dir / "raw_artifacts_emg.fif"

# пути для ручного запуска
# fif_path = Path(r"/Users/user/Desktop/raw_artifacts_emg.fif")
# fif_path = Path(r"C:\\Users\\user\\Desktop\\raw_artifacts_emg.fif")

recording_name = (
    fif_path.name
    .removesuffix(".fif.gz")
    .removesuffix(".fif")
)
annotations_path = qc_dir / f"{recording_name}-annot.fif"
legacy_annotations_path = qc_dir / f"{recording_name}_annotations.csv"

print("Project root:", project_root)
print("Input FIF:", fif_path)
print("Annotations:", annotations_path)
print("Interim:", data_interim_dir)
print("Processed:", data_processed_dir)
print("Figures:", figures_dir)
print("QC:", qc_dir)
print("Tables:", tables_dir)


## загрузка данных

In [ ]:
if not fif_path.exists():
    raise FileNotFoundError(f"Файл не найден: {fif_path}")

raw_original = mne.io.read_raw_fif(fif_path, preload=True)
raw_original.set_channel_types({
    channel: "emg" for channel in raw_original.ch_names
})
raw_base = raw_original.copy()

# параметры влияют только на отображение, данные остаются в вольтах
emg_browser_kwargs = dict(
    duration=1.0,
    n_channels=8,
    scalings={"emg": 4e-3},
)

# восстанавливаем сохранённую разметку при повторном запуске
if annotations_path.exists():
    saved_annotations = mne.read_annotations(annotations_path)
    raw_base.set_annotations(saved_annotations)
    annotations_source = annotations_path
elif legacy_annotations_path.exists():
    # преобразуем onset старого CSV из времени Unix в секунды записи
    annotations_table = pd.read_csv(legacy_annotations_path)
    epoch = pd.Timestamp("1970-01-01", tz="UTC")
    annotation_onsets = (
        pd.to_datetime(annotations_table["onset"], utc=True) - epoch
    ).dt.total_seconds().to_numpy()
    saved_annotations = mne.Annotations(
        onset=annotation_onsets,
        duration=annotations_table["duration"].to_numpy(),
        description=annotations_table["description"].astype(str).to_numpy(),
        orig_time=None,
    )
    raw_base.set_annotations(saved_annotations)
    annotations_source = legacy_annotations_path
else:
    annotations_source = None

sfreq = raw_base.info["sfreq"]
duration_s = raw_base.n_times / sfreq

print("File:", fif_path.name)
print("Channels:", len(raw_base.ch_names))
print("sfreq:", sfreq)
print("Duration, s:", round(duration_s, 2))
print("First channels:", raw_base.ch_names[:10])
print("Annotations:", len(raw_base.annotations))
print("Annotations loaded from:", annotations_source or "input FIF")


## проверка записи

In [ ]:
if "TA L" not in raw_base.info["bads"]:
    raw_base.info["bads"].append("TA L")

channels_table = pd.DataFrame(
    {
        "channel": raw_base.ch_names,
        "type": raw_base.get_channel_types(),
        "bad": [ch in raw_base.info["bads"] for ch in raw_base.ch_names],
    }
)

display(channels_table)
print(f"Sampling rate: {raw_base.info['sfreq']} Hz")
print(f"Duration: {duration_s:.2f} s")
print("Bad channels:", raw_base.info["bads"] or "none")

raw_base.plot(**emg_browser_kwargs)


## сохранение аннотаций (опционально, фильтрация их не использует)

In [ ]:

raw_base.annotations.save(
    annotations_path,
    overwrite=True,
)

print(f"Сохранено аннотаций: {len(raw_base.annotations)}")
print("Файл:", annotations_path)

## выбор каналов

In [ ]:
emg_channels = ["GM R", "GM L", "RF R", "RF L", "TA R", "TA L", "BF R", "BF L"]
right_channels = ["GM R", "RF R", "TA R", "BF R"]
left_channels = ["GM L", "RF L", "TA L", "BF L"]

excluded_channels = ["TA L"]

print("EMG channels:", emg_channels)
print("Right:", right_channels)
print("Left:", left_channels)
print("Excluded:", excluded_channels)


## CAR глобальный

In [ ]:
car_channels = [
    channel for channel in emg_channels
    if channel not in excluded_channels
]

raw_car_global = raw_base.copy()
reference_signal = raw_car_global.get_data(picks=car_channels).mean(axis=0)
channel_indices = [
    raw_car_global.ch_names.index(channel)
    for channel in car_channels
]
raw_car_global._data[channel_indices] -= reference_signal

car_global_path = data_interim_dir / "recording_car_global_raw.fif"
raw_car_global.save(car_global_path, overwrite=True)

print("Saved:", car_global_path)
raw_car_global.plot(**emg_browser_kwargs)

## CAR по сторонам

In [ ]:
raw_car_left_right = raw_base.copy()

for side_channels in [right_channels, left_channels]:
    car_channels = [
        channel for channel in side_channels
        if channel not in excluded_channels
    ]
    reference_signal = raw_car_left_right.get_data(
        picks=car_channels
    ).mean(axis=0)
    channel_indices = [
        raw_car_left_right.ch_names.index(channel)
        for channel in car_channels
    ]
    raw_car_left_right._data[channel_indices] -= reference_signal

car_left_right_path = (
    data_interim_dir / "recording_car_left_right_raw.fif"
)
raw_car_left_right.save(car_left_right_path, overwrite=True)

print("Saved:", car_left_right_path)
raw_car_left_right.plot(**emg_browser_kwargs)

## подготовка методов

In [ ]:
analysis_channels = [
    channel for channel in emg_channels
    if channel not in excluded_channels
]

missing_channels = [
    channel for channel in analysis_channels
    if channel not in raw_base.ch_names
]
if missing_channels:
    raise ValueError(f"В записи отсутствуют каналы: {missing_channels}")

print("Analysis channels:", analysis_channels)
print("Excluded channels:", excluded_channels)

### notch-фильтр 50 Гц

Все отсчёты от начала до конца записи используются для обучения, применения и метрик. Медленная часть (≤ 30 Гц) считается артефактом, нерегулярная быстрая часть — полезным ЭМГ. Регулярная сетевая компонента 50 Гц удаляется узким notch-фильтром.

In [ ]:
LINE_FREQ_HZ = 50.0
LINE_NOTCH_Q = 30.0

emg_data = raw_base.get_data(picks=analysis_channels)
notch_b, notch_a = signal.iirnotch(
    LINE_FREQ_HZ,
    LINE_NOTCH_Q,
    fs=sfreq,
)
notched_emg_data = signal.filtfilt(notch_b, notch_a, emg_data, axis=1)

raw_notch = raw_base.copy().load_data()
notch_indices = [
    raw_notch.ch_names.index(channel)
    for channel in analysis_channels
]
raw_notch._data[notch_indices] = notched_emg_data

print("Notch frequency, Hz:", LINE_FREQ_HZ)
print("Notch Q:", LINE_NOTCH_Q)
raw_notch.plot(**emg_browser_kwargs)

### непрерывные данные и частотное разделение

In [ ]:
ARTIFACT_CUTOFF_HZ = 30.0  # медленнее: артефакт, быстрее: ЭМГ

lowpass_filter = signal.butter(
    4,
    ARTIFACT_CUTOFF_HZ,
    btype="lowpass",
    fs=sfreq,
    output="sos",
)
slow_artifact_data = signal.sosfiltfilt(
    lowpass_filter,
    emg_data,
    axis=1,
)
retained_emg_data = emg_data - slow_artifact_data

assert slow_artifact_data.shape == emg_data.shape
assert np.allclose(
    slow_artifact_data + retained_emg_data,
    emg_data,
)
print("Continuous training duration, s:", round(raw_base.n_times / sfreq, 3))
print("Slow-artifact cutoff, Hz:", ARTIFACT_CUTOFF_HZ)

## SVD rank-1 по медленной части всей записи

In [ ]:
svd_covariance = np.cov(slow_artifact_data)
left_vectors, svd_singular_values, _ = np.linalg.svd(
    svd_covariance, full_matrices=False
)
artifact_direction = left_vectors[:, :1]  # главная компонента

# строим ортогональный проектор для удаления главной компоненты
svd_projector = (
    np.eye(len(analysis_channels))
    - artifact_direction @ artifact_direction.T
)

# удаляем rank-1 модель медленного артефакта на всей записи
svd_artifact_model = artifact_direction @ (artifact_direction.T @ slow_artifact_data)
svd_clean_data = emg_data - svd_artifact_model
raw_svd = raw_base.copy().load_data()
svd_indices = [
    raw_svd.ch_names.index(channel)
    for channel in analysis_channels
]
raw_svd._data[svd_indices] = svd_clean_data

svd_explained_fraction = (
    svd_singular_values[0] / svd_singular_values.sum()
)
print("First SVD covariance fraction:", round(svd_explained_fraction, 4))
print("Projector symmetric:", np.allclose(svd_projector, svd_projector.T))
print(
    "Projector idempotent:",
    np.allclose(svd_projector @ svd_projector, svd_projector),
)
print("SVD applied samples:", svd_clean_data.shape[1])
assert svd_clean_data.shape == emg_data.shape
assert np.isfinite(svd_clean_data).all()

raw_svd.plot(**emg_browser_kwargs)

## GED: медленный артефакт против быстрого ЭМГ по всей записи

### параметры

In [ ]:
GED_REGULARIZATION = 0.05  # регуляризация ковариации
GED_MIN_POWER_RATIO = 3.0  # порог мощности

ged_model_path = qc_dir / "ged_slow_global_model.npz"
ged_output_path = data_interim_dir / "recording_ged_global_raw.fif"

print("Slow-artifact cutoff, Hz:", ARTIFACT_CUTOFF_HZ)

### область обучения и применения

In [ ]:
training_samples = raw_base.n_times
application_samples = raw_base.n_times
print("GED training samples:", training_samples)
print("GED application samples:", application_samples)

### модель артефакта

In [ ]:
# межканальная ковариация
def covariance(data):
    centered_data = data - data.mean(axis=1, keepdims=True)
    sample_count = centered_data.shape[1]
    return centered_data @ centered_data.T / (sample_count - 1)


artifact_covariance = covariance(slow_artifact_data)
normal_covariance = covariance(retained_emg_data)
regularization = (
    GED_REGULARIZATION
    * np.trace(normal_covariance)
    / len(analysis_channels)
)
regularized_normal_covariance = (
    normal_covariance
    + regularization * np.eye(len(analysis_channels))
)

power_ratios, spatial_filters = linalg.eigh(
    artifact_covariance,
    regularized_normal_covariance,
)
selected_components = power_ratios > GED_MIN_POWER_RATIO
if not selected_components.any():
    raise ValueError("GED не нашёл компонент, специфичных для артефакта")

ged_filters = spatial_filters[:, selected_components]
component_signals = ged_filters.T @ slow_artifact_data

# возвращаем компоненты в пространство каналов для построения модели артефакта
spatial_patterns = (
    slow_artifact_data
    @ component_signals.T
    @ np.linalg.pinv(
        component_signals @ component_signals.T
    )
)
artifact_model = spatial_patterns @ component_signals

eigenvalue_table = pd.DataFrame({
    "slow artifact / retained EMG power": power_ratios[::-1],
    "remove": selected_components[::-1],
})
display(eigenvalue_table)
display(pd.DataFrame(
    spatial_patterns,
    index=analysis_channels,
    columns=[
        f"component {index + 1}"
        for index in range(selected_components.sum())
    ],
))

### вычитание компоненты

In [ ]:
clean_emg_data = emg_data - artifact_model
raw_ged = raw_base.copy().load_data()
ged_indices = [
    raw_ged.ch_names.index(channel)
    for channel in analysis_channels
]
raw_ged._data[ged_indices] = clean_emg_data

print("GED applied samples:", clean_emg_data.shape[1])
assert clean_emg_data.shape == emg_data.shape
assert np.isfinite(clean_emg_data).all()

raw_ged.plot(**emg_browser_kwargs)

## проверка сигнала

In [ ]:
records = {
    "original": raw_base,
    "50 Hz notch": raw_notch,
    "global CAR": raw_car_global,
    "left/right CAR": raw_car_left_right,
    "SVD slow rank-1": raw_svd,
    "GED slow components": raw_ged,
}

# среднеквадратичная амплитуда
def rms(data, axis=-1):
    return np.sqrt(np.mean(np.square(data), axis=axis))


rms_table = pd.DataFrame({
    name: rms(record.get_data(picks=analysis_channels), axis=1)
    for name, record in records.items()
}, index=analysis_channels)
rms_table.index.name = "channel"
display(rms_table)

clean_slow_artifact_data = signal.sosfiltfilt(
    lowpass_filter,
    clean_emg_data,
    axis=1,
)
clean_retained_emg_data = clean_emg_data - clean_slow_artifact_data

slow_artifact_rms = rms(slow_artifact_data, axis=1)
clean_slow_artifact_rms = rms(clean_slow_artifact_data, axis=1)
retained_emg_rms = rms(retained_emg_data, axis=1)
clean_retained_emg_rms = rms(clean_retained_emg_data, axis=1)

quality = pd.DataFrame({
    "slow artifact RMS after / before": (
        clean_slow_artifact_rms / slow_artifact_rms
    ),
    "retained EMG RMS after / before": (
        clean_retained_emg_rms / retained_emg_rms
    ),
    "slow artifact / EMG before": slow_artifact_rms / retained_emg_rms,
    "slow artifact / EMG after": (
        clean_slow_artifact_rms / clean_retained_emg_rms
    ),
}, index=analysis_channels)
quality.index.name = "channel"
display(quality)

## визуальное сравнение

In [ ]:
plot_channel = analysis_channels[0]
max_plot_points = 20_000
plot_step = max(1, raw_base.n_times // max_plot_points)
time = raw_base.times[::plot_step]

fig, axes = plt.subplots(
    len(records), 1, figsize=(14, 2.2 * len(records)), sharex=True, sharey=True
)
for axis, (name, record) in zip(axes, records.items()):
    values = record.get_data(picks=[plot_channel])[0, ::plot_step]
    axis.plot(time, values, linewidth=0.9)
    axis.set_title(name)
    axis.set_ylabel("V")
axes[-1].set_xlabel("Time, s")
fig.suptitle(f"{plot_channel}: complete recording")
fig.tight_layout()
plt.show()


## сохранение результатов

In [ ]:
notch_output_path = data_interim_dir / "recording_notch_50hz_raw.fif"
notch_model_path = qc_dir / "notch_50hz_filter.npz"
raw_notch.save(notch_output_path, overwrite=True)
raw_ged.save(ged_output_path, overwrite=True)
svd_output_path = data_interim_dir / "recording_svd_k1_global_raw.fif"
svd_model_path = qc_dir / "svd_k1_slow_global_model.npz"
raw_svd.save(svd_output_path, overwrite=True)

np.savez(
    ged_model_path,
    channels=np.asarray(analysis_channels),
    covariance_artifact=artifact_covariance,
    covariance_normal=normal_covariance,
    eigenvalues=power_ratios,
    ged_filters=ged_filters,
    spatial_patterns=spatial_patterns,
    artifact_cutoff_hz=ARTIFACT_CUTOFF_HZ,
    training_samples=training_samples,
)

np.savez(
    svd_model_path,
    channels=np.asarray(analysis_channels),
    covariance=svd_covariance,
    singular_values=svd_singular_values,
    artifact_direction=artifact_direction,
    projector=svd_projector,
    artifact_cutoff_hz=ARTIFACT_CUTOFF_HZ,
    training_samples=training_samples,
)

np.savez(
    notch_model_path,
    channels=np.asarray(analysis_channels),
    line_freq_hz=LINE_FREQ_HZ,
    line_notch_q=LINE_NOTCH_Q,
    numerator=notch_b,
    denominator=notch_a,
)

output_files = {
    "50 Hz notch": notch_output_path,
    "global CAR": car_global_path,
    "left/right CAR": car_left_right_path,
    "SVD slow rank-1": svd_output_path,
    "GED slow components": ged_output_path,
}
for name, path in output_files.items():
    print(f"{name}: {path}")
print("Notch model:", notch_model_path)
print("GED global model:", ged_model_path)
print("SVD global model:", svd_model_path)

## сравнение методов

### RMS

In [ ]:
comparison_channels = analysis_channels
epsilon = np.finfo(float).eps

before_data = raw_base.get_data(picks=comparison_channels)
before_notched = signal.filtfilt(notch_b, notch_a, before_data, axis=1)
before_slow_artifact = signal.sosfiltfilt(
    lowpass_filter, before_notched, axis=1
)
before_retained_emg = before_notched - before_slow_artifact
whole_rms_before = rms(before_data, axis=1)
slow_artifact_rms_before = rms(before_slow_artifact, axis=1)
retained_emg_rms_before = rms(before_retained_emg, axis=1)

def line_band_power(data, center_hz=LINE_FREQ_HZ, half_width_hz=0.5):
    frequencies, psd = signal.welch(
        data,
        fs=sfreq,
        nperseg=min(data.shape[1], int(round(8 * sfreq))),
        axis=1,
    )
    line_bins = np.abs(frequencies - center_hz) <= half_width_hz
    frequency_step = frequencies[1] - frequencies[0]
    return psd[:, line_bins].sum(axis=1) * frequency_step

line_power_before = line_band_power(before_data)

metric_rows = []
channel_metrics = {}

for method, record in records.items():
    after_data = record.get_data(picks=comparison_channels)
    after_notched = signal.filtfilt(notch_b, notch_a, after_data, axis=1)
    after_slow_artifact = signal.sosfiltfilt(
        lowpass_filter, after_notched, axis=1
    )
    after_retained_emg = after_notched - after_slow_artifact
    whole_rms = rms(after_data, axis=1)
    slow_artifact_rms_after = rms(after_slow_artifact, axis=1)
    retained_emg_rms_after = rms(after_retained_emg, axis=1)
    line_power_after = line_band_power(after_data)
    values = {
        "Whole-recording RMS": whole_rms,
        "Whole-recording RMS ratio": whole_rms / (whole_rms_before + epsilon),
        "Slow artifact RMS ratio": (
            slow_artifact_rms_after / (slow_artifact_rms_before + epsilon)
        ),
        "Retained EMG RMS ratio": (
            retained_emg_rms_after / (retained_emg_rms_before + epsilon)
        ),
        "50 Hz power ratio": line_power_after / (line_power_before + epsilon),
        "Slow artifact / retained EMG": (
            slow_artifact_rms_after / (retained_emg_rms_after + epsilon)
        ),
    }
    channel_metrics[method] = pd.DataFrame(
        values,
        index=comparison_channels,
    )

    for metric, vector in values.items():
        q1, median, q3 = np.nanpercentile(vector, [25, 50, 75])
        metric_rows.append({
            "method": method,
            "metric": metric,
            "median": median,
            "q1": q1,
            "q3": q3,
        })

metrics_long = pd.DataFrame(metric_rows)

# форматируем медиану и межквартильный интервал
def format_metric(row):
    return f"{row['median']:.3f} [{row['q1']:.3f}–{row['q3']:.3f}]"


metrics_display = metrics_long.assign(
    value=metrics_long.apply(format_metric, axis=1)
).pivot(
    index="metric",
    columns="method",
    values="value",
)
metric_order = list(channel_metrics["original"].columns)
display(metrics_display.reindex(metric_order))

### сравнение до и после

In [ ]:
plot_channel = "TA R"  # канал для графика
method_records = {name: rec for name, rec in records.items() if name != "original"}
max_plot_points = 20_000
plot_step = max(1, raw_base.n_times // max_plot_points)
t = raw_base.times[::plot_step]
before_view = raw_base.get_data(picks=[plot_channel])[0, ::plot_step]
after_views = {
    name: rec.get_data(picks=[plot_channel])[0, ::plot_step]
    for name, rec in method_records.items()
}
all_signals = np.concatenate([before_view, *after_views.values()])
signal_limit = 1.05 * np.max(np.abs(all_signals))

fig, axes = plt.subplots(len(method_records), 1, figsize=(16, 3.1 * len(method_records)), sharex=True, sharey=True)
axes = np.atleast_1d(axes)
for ax, (method, after_view) in zip(axes, after_views.items()):
    ax.plot(t, before_view, color="#E63946", lw=1.6, alpha=0.95, label="before", zorder=3)
    ax.plot(t, after_view, color="#0066FF", lw=1.35, alpha=0.95, label="after", zorder=4)
    ax.set_ylim(-signal_limit, signal_limit)
    ax.set_ylabel("Amplitude, V")
    ax.set_title(method, loc="left", fontweight="bold")
    ax.grid(alpha=0.18)
axes[0].legend(loc="upper right", frameon=True, ncol=2)
axes[-1].set_xlabel("Time, s")
fig.suptitle(f"{plot_channel}: before/after over complete recording", y=1.01)
fig.tight_layout()
plt.show()


### отношения RMS

In [ ]:
method_order = list(records)
artifact_heat = pd.DataFrame({m: channel_metrics[m]["Slow artifact RMS ratio"] for m in method_order}).T
normal_heat = pd.DataFrame({m: channel_metrics[m]["Retained EMG RMS ratio"] for m in method_order}).T

fig, axes = plt.subplots(1, 2, figsize=(18, 5), constrained_layout=True)
for ax, table, title, cmap, limits in [
    (axes[0], artifact_heat, "Slow artifact RMS: after / before", "viridis", (0, max(1, np.nanpercentile(artifact_heat, 95)))),
    (axes[1], normal_heat, "Retained EMG RMS: after / before (идеал = 1)", "coolwarm", (0.5, 1.5)),
]:
    image = ax.imshow(table, aspect="auto", cmap=cmap, vmin=limits[0], vmax=limits[1])
    ax.set_xticks(range(len(table.columns)), table.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(table.index)), table.index)
    ax.set_title(title)
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            ax.text(j, i, f"{table.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8, color="black")
    fig.colorbar(image, ax=ax, shrink=0.85)
plt.show()


### межканальные корреляции

In [ ]:
corr_matrices = {
    method: np.corrcoef(record.get_data(picks=comparison_channels))
    for method, record in records.items()
}
fig, axes = plt.subplots(1, len(corr_matrices), figsize=(4.2 * len(corr_matrices), 4.3), constrained_layout=True)
for ax, (method, matrix) in zip(axes, corr_matrices.items()):
    image = ax.imshow(matrix, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_title(method)
    ax.set_xticks(range(len(comparison_channels)), comparison_channels, rotation=90)
    ax.set_yticks(range(len(comparison_channels)), comparison_channels)
fig.colorbar(image, ax=axes, shrink=0.75, label="Pearson r")
fig.suptitle("Межканальная корреляция по всей записи")
plt.show()

# вычитаем исходную матрицу для оценки изменения корреляций
fig, axes = plt.subplots(1, len(corr_matrices) - 1, figsize=(4.2 * (len(corr_matrices) - 1), 4.3), constrained_layout=True)
delta_limit = max(np.max(np.abs(m - corr_matrices["original"])) for name, m in corr_matrices.items() if name != "original")
for ax, (method, matrix) in zip(axes, [(n, m) for n, m in corr_matrices.items() if n != "original"]):
    delta = matrix - corr_matrices["original"]
    image = ax.imshow(delta, cmap="coolwarm", vmin=-delta_limit, vmax=delta_limit)
    ax.set_title(method)
    ax.set_xticks(range(len(comparison_channels)), comparison_channels, rotation=90)
    ax.set_yticks(range(len(comparison_channels)), comparison_channels)
fig.colorbar(image, ax=axes, shrink=0.75, label=r"$\Delta$ Pearson r")
fig.suptitle("Изменение корреляции относительно original")
plt.show()


## просмотр результата